# [dbo].[sp_CapNhatTrangThaiThanhToan]

In [0]:
USE [PetCareX_DB]
GO


In [0]:
/****** Object:  StoredProcedure [dbo].[sp_CapNhatTrangThaiThanhToan]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- Stored Procedure 6: Cập nhật trạng thái thanh toán
CREATE PROCEDURE [dbo].[sp_CapNhatTrangThaiThanhToan]
    @MaHD VARCHAR(20)
AS
BEGIN
    SET NOCOUNT ON;
    
    BEGIN TRY
        BEGIN TRANSACTION;
        
        UPDATE HOA_DON
        SET TrangThai = 'DaThanhToan'
        WHERE MaHD = @MaHD;
        
        IF @@ROWCOUNT = 0
        BEGIN
            ROLLBACK TRANSACTION;
            SELECT 0 AS Result, N'Không tìm thấy hóa đơn' AS Message;
            RETURN;
        END
        
        COMMIT TRANSACTION;
        SELECT 1 AS Result, N'Thanh toán thành công' AS Message;
    END TRY
    BEGIN CATCH
        IF @@TRANCOUNT > 0
            ROLLBACK TRANSACTION;
            
        SELECT 0 AS Result, ERROR_MESSAGE() AS Message;
    END CATCH
END

GO


# [dbo].[sp_ChamCong_CheckIn]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_ChamCong_CheckIn]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- =============================================
-- 1. CHECK-IN
-- Logic: 
-- - Lấy ngày giờ hiện tại của Server.
-- - Nếu đã Check-in hôm nay rồi thì báo lỗi (để không bị ghi đè giờ cũ)
-- =============================================
CREATE   PROCEDURE [dbo].[sp_ChamCong_CheckIn]
    @MaNV VARCHAR(20)
AS
BEGIN
    SET NOCOUNT ON;
    
    DECLARE @NgayHienTai DATE = CAST(GETDATE() AS DATE);
    DECLARE @GioHienTai TIME = CAST(GETDATE() AS TIME);

    -- Kiểm tra nhân viên tồn tại
    IF NOT EXISTS (SELECT 1 FROM NHAN_VIEN WHERE MaNV = @MaNV)
    BEGIN
        ;THROW 52001, N'Lỗi: Mã nhân viên không tồn tại.', 1;
        RETURN;
    END

    -- Kiểm tra xem hôm nay đã Check-in chưa
    IF EXISTS (SELECT 1 FROM CHAM_CONG WHERE MaNV = @MaNV AND NgayLamViec = @NgayHienTai)
    BEGIN
        ;THROW 52002, N'Bạn đã Check-in ngày hôm nay rồi.', 1;
        RETURN;
    END

    -- Thực hiện Check-in
    INSERT INTO CHAM_CONG (MaNV, NgayLamViec, Checkin, Checkout)
    VALUES (@MaNV, @NgayHienTai, @GioHienTai, NULL);
    
    SELECT N'Check-in thành công lúc ' + CAST(@GioHienTai AS NVARCHAR(20)) AS ThongBao;
END

GO


# [dbo].[sp_ChamCong_CheckOut]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_ChamCong_CheckOut]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- =============================================
-- 2. CHECK-OUT
-- Logic: 
-- - Phải có dữ liệu Check-in hôm nay thì mới được Check-out
-- - Update giờ Check-out vào dòng dữ liệu của ngày hôm nay
-- =============================================
CREATE   PROCEDURE [dbo].[sp_ChamCong_CheckOut]
    @MaNV VARCHAR(20)
AS
BEGIN
    SET NOCOUNT ON;

    DECLARE @NgayHienTai DATE = CAST(GETDATE() AS DATE);
    DECLARE @GioHienTai TIME = CAST(GETDATE() AS TIME);

    -- 1. Kiểm tra xem đã Check-in hôm nay chưa
    IF NOT EXISTS (SELECT 1 FROM CHAM_CONG WHERE MaNV = @MaNV AND NgayLamViec = @NgayHienTai)
    BEGIN
        ;THROW 52003, N'Bạn chưa Check-in, không thể Check-out.', 1;
        RETURN;
    END
	
	-- 2. Kiểm tra Check-out đã tồn tại chưa
	IF EXISTS (SELECT 1 FROM CHAM_CONG WHERE MaNV = @MaNV AND NgayLamViec = @NgayHienTai AND (Checkout IS NOT NULL))
    BEGIN
        ;THROW 52002, N'Bạn đã Check-OUT ngày hôm nay rồi.', 1;
        RETURN;
    END

    -- 3. Thực hiện Check-out (Update dòng cũ)
    UPDATE CHAM_CONG
    SET Checkout = @GioHienTai
    WHERE MaNV = @MaNV AND NgayLamViec = @NgayHienTai;

    SELECT N'Check-out thành công lúc ' + CAST(@GioHienTai AS NVARCHAR(20)) AS ThongBao;
END

GO


# [dbo].[sp_ChamCong_LayTrangThai]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_ChamCong_LayTrangThai]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- Kiểm tra trạng thái chấm công
CREATE   PROCEDURE [dbo].[sp_ChamCong_LayTrangThai]
    @MaNV VARCHAR(20)
AS
BEGIN
    SET NOCOUNT ON;
    DECLARE @NgayHienTai DATE = CAST(GETDATE() AS DATE);

    SELECT 
        MaNV,
        NgayLamViec,
        Checkin,
        Checkout,
        CASE 
            WHEN Checkin IS NULL THEN N'Chưa vào làm'
            WHEN Checkout IS NULL THEN N'Đang làm việc'
            ELSE N'Đã ra về'
        END AS TrangThai
    FROM CHAM_CONG
    WHERE MaNV = @MaNV AND NgayLamViec = @NgayHienTai;
END

GO


# [dbo].[sp_ChamCong_XemLichSu]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_ChamCong_XemLichSu]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-------
------ Xóa các record chấm công trong ngày để test CHECK-IN và CHECK-OUT
-------

-- Xem lịch sử chấm công trong 30 ngày gần nhất
CREATE   PROCEDURE [dbo].[sp_ChamCong_XemLichSu]
    @MaNV VARCHAR(20)
AS
BEGIN
    SET NOCOUNT ON;

    SELECT TOP 31
        cc.NgayLamViec,
        nv.MaNV,
        nv.HoTen,

        CAST(cc.Checkin AS time(0))  AS Checkin,
        CAST(cc.Checkout AS time(0)) AS Checkout,

        CAST(
            CASE 
                WHEN cc.Checkin IS NOT NULL AND cc.Checkout IS NOT NULL 
                THEN DATEDIFF(MINUTE, cc.Checkin, cc.Checkout) / 60.0
                ELSE 0
            END
        AS DECIMAL(5,2)) AS SoGioLamViec

    FROM CHAM_CONG cc
    JOIN NHAN_VIEN nv ON cc.MaNV = nv.MaNV
    WHERE cc.MaNV = @MaNV
    ORDER BY cc.NgayLamViec DESC, nv.HoTen ASC;
END

GO


# [dbo].[sp_CheckProductStock]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_CheckProductStock]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- 6. Kiểm tra Tồn Kho Sản Phẩm
--DROP PROCEDURE sp_CheckProductStock
CREATE PROCEDURE [dbo].[sp_CheckProductStock]
    @MaSP VARCHAR(20),
    @MaCN VARCHAR(20)
AS
BEGIN
    SELECT 
        ISNULL(SUM(SoLuong), 0) AS TonKho
    FROM TON_KHO_SAN_PHAM
    WHERE MaSP = @MaSP AND MaCN = @MaCN
END
GO


# [dbo].[sp_CreateCustomer]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_CreateCustomer]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- 5. Tạo Khách Hàng Mới
--DROP PROCEDURE sp_CreateCustomer
CREATE PROCEDURE [dbo].[sp_CreateCustomer]
    @MaKH VARCHAR(20),
    @HoTen NVARCHAR(100),
    @SoDT NVARCHAR(100),
    @Email NVARCHAR(100),
    @GioiTinh NVARCHAR(100)
AS
BEGIN
    BEGIN TRY
        INSERT INTO KHACH_HANG (MaKH, HoTen, SoDT, Email, GioiTinh, CapHoiVien, DiemTichLuy)
        VALUES (@MaKH, @HoTen, @SoDT, @Email, @GioiTinh, 'CoBan', 0)
        
        SELECT 
            MaKH,
            HoTen,
            SoDT,
            Email,
            GioiTinh,
            CapHoiVien,
            DiemTichLuy
        FROM KHACH_HANG
        WHERE MaKH = @MaKH
    END TRY
    BEGIN CATCH
        THROW
    END CATCH
END
GO


# [dbo].[sp_CreateInvoice]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_CreateInvoice]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- 7. Tạo Hóa Đơn (Transaction)
--DROP PROCEDURE sp_CreateInvoice
CREATE PROCEDURE [dbo].[sp_CreateInvoice]
(
    @MaHD VARCHAR(20),
    @NgayLap DATE,
    @TongTien DECIMAL(18,0),
    @KhuyenMai NVARCHAR(100),
    @HinhThucThanhToan NVARCHAR(100),
    @MaNV VARCHAR(20),
    @MaKH VARCHAR(20),
    @MaCN VARCHAR(20),
    @MaMuaHang VARCHAR(20),
    @InvoiceDetails NVARCHAR(MAX)   -- JSON
)
AS
BEGIN
    SET NOCOUNT ON;

    BEGIN TRY
        
        IF @@TRANCOUNT = 0 
            BEGIN TRAN
        ELSE 
            SAVE TRAN sp_SavePoint   -- nếu lỡ gọi lồng nhau

        ----------------------------------------------------
        -- 1. Tạo hóa đơn
        ----------------------------------------------------
        INSERT INTO dbo.HOA_DON (MaHD, NgayLap, TongTien, KhuyenMai, HinhThucThanhToan, TrangThai, MaNV, MaKH, MaCN)
        VALUES (@MaHD, @NgayLap, @TongTien, @KhuyenMai, @HinhThucThanhToan, N'ChuaThanhToan', @MaNV, @MaKH, @MaCN);

        ----------------------------------------------------
        -- 2. Tạo dịch vụ
        ----------------------------------------------------
        INSERT INTO dbo.DICH_VU (MaDV, MaTC, MaCN)
        VALUES (@MaMuaHang, NULL, @MaCN);

        ----------------------------------------------------
        -- 3. Tạo dịch vụ mua hàng
        ----------------------------------------------------
        INSERT INTO dbo.DV_MUA_HANG (MaMuaHang, NhanVienBanHang)
        VALUES (@MaMuaHang, @MaNV);

        ----------------------------------------------------
        -- 4. Liên kết hóa đơn – dịch vụ
        ----------------------------------------------------
        INSERT INTO dbo.CHI_TIET_DV_SD (MaHD, MaDV)
        VALUES (@MaHD, @MaMuaHang);

        ----------------------------------------------------
        -- 5. Thêm chi tiết mua hàng từ JSON
        ----------------------------------------------------
        INSERT INTO dbo.CHI_TIET_MUA_HANG (MaMuaHang, MaSP, SoLuong, DonGia)
        SELECT 
            @MaMuaHang,
            JSON_VALUE(value, '$.MaSP'),
            CAST(JSON_VALUE(value, '$.SoLuong') AS INT),
            CAST(JSON_VALUE(value, '$.DonGia') AS DECIMAL(18,0))
        FROM OPENJSON(@InvoiceDetails);

        ----------------------------------------------------
        -- 6. Trừ tồn kho
        ----------------------------------------------------
        UPDATE tk
        SET tk.SoLuong = tk.SoLuong - d.SoLuong
        FROM dbo.TON_KHO_SAN_PHAM tk
        INNER JOIN (
            SELECT 
                JSON_VALUE(value, '$.MaSP') AS MaSP,
                CAST(JSON_VALUE(value, '$.SoLuong') AS INT) AS SoLuong
            FROM OPENJSON(@InvoiceDetails)
        ) d ON tk.MaSP = d.MaSP
        WHERE tk.MaCN = @MaCN;

        ----------------------------------------------------
        -- 7. Không cho âm kho
        ----------------------------------------------------
        IF EXISTS (
            SELECT 1 
            FROM dbo.TON_KHO_SAN_PHAM 
            WHERE MaCN = @MaCN AND SoLuong < 0
        )
        BEGIN
            THROW 50001, N'Không đủ tồn kho cho một số sản phẩm', 1;
        END

        ----------------------------------------------------
        -- 8. Commit an toàn
        ----------------------------------------------------
        IF @@TRANCOUNT > 0
            COMMIT;

        SELECT 'SUCCESS' AS Status, @MaHD AS MaHD;
    END TRY

    BEGIN CATCH
        
        IF @@TRANCOUNT > 0
            ROLLBACK;

        THROW;
    END CATCH
END
GO


# [dbo].[sp_GetChiNhanhByMa]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_GetChiNhanhByMa]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- 9. Lấy thông tin Chi Nhánh theo Mã
--DROP PROCEDURE sp_GetChiNhanhByMa
CREATE PROCEDURE [dbo].[sp_GetChiNhanhByMa]
    @MaCN VARCHAR(20)
AS
BEGIN
    SELECT MaCN, TenChiNhanh, DiaChi, SoDienThoai
    FROM CHI_NHANH
    WHERE MaCN = @MaCN
END
GO


# [dbo].[sp_GetCustomerByMa]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_GetCustomerByMa]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- 10. Lấy thông tin Khách Hàng theo Mã
--DROP PROCEDURE sp_GetCustomerByMa
CREATE PROCEDURE [dbo].[sp_GetCustomerByMa]
    @MaKH VARCHAR(20)
AS
BEGIN
    SELECT 
        MaKH,
        HoTen,
        SoDT,
        Email,
        GioiTinh,
        CapHoiVien,
        ISNULL(DiemTichLuy, 0) AS DiemTichLuy
    FROM KHACH_HANG
    WHERE MaKH = @MaKH
END
GO


# [dbo].[sp_GetDanhSachBacSi]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_GetDanhSachBacSi]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- 2. Lấy danh sách Bác sĩ (Chỉ lấy Mã và Tên để đổ vào ComboBox)
CREATE   PROCEDURE [dbo].[sp_GetDanhSachBacSi]
AS
BEGIN
    SELECT MaNV, HoTen 
    FROM NHAN_VIEN 
    WHERE ChucVu = 'BacSi'
END

GO


# [dbo].[sp_GetDanhSachChiNhanh]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_GetDanhSachChiNhanh]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- 3. Lấy danh sách Chi nhánh (Chỉ lấy Mã và Tên)
CREATE   PROCEDURE [dbo].[sp_GetDanhSachChiNhanh]
AS
BEGIN
    SELECT MaCN, TenChiNhanh FROM CHI_NHANH
END

GO


# [dbo].[sp_GetDanhSachLichHenKham]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_GetDanhSachLichHenKham]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- 6. SP Lấy danh sách lịch hẹn khám bệnh (cho form Lịch Hẹn)
CREATE   PROCEDURE [dbo].[sp_GetDanhSachLichHenKham]
    @TrangThai NVARCHAR(50) = NULL,
    @TuNgay DATE = NULL,
    @DenNgay DATE = NULL
AS
BEGIN
    -- Lấy tất cả nếu tham số NULL
    SELECT 
        l.MaLichHen,
        l.NgayHen,
        l.GioHen,
        l.TrangThai,
        l.GhiChu,
        ISNULL(k.HoTen, N'Khách vãng lai') AS TenKhachHang,
        ISNULL(k.SoDT, '') AS SoDT,
        ISNULL(bs.HoTen, '') AS TenBacSi,
        ISNULL(l.MaBS, '') AS MaBacSi, -- Cần mã BS để truyền sang form Khám
        ISNULL(cn.TenChiNhanh, '') AS TenChiNhanh
    FROM LICH_HEN l
    LEFT JOIN KHACH_HANG k ON l.MaKH = k.MaKH
    LEFT JOIN NHAN_VIEN bs ON l.MaBS = bs.MaNV
    LEFT JOIN CHI_NHANH cn ON l.MaCN = cn.MaCN
    WHERE 
        (@TrangThai IS NULL OR l.TrangThai = @TrangThai OR @TrangThai = '')
        AND (@TuNgay IS NULL OR l.NgayHen >= @TuNgay)
        AND (@DenNgay IS NULL OR l.NgayHen <= @DenNgay)
    ORDER BY l.NgayHen DESC, l.GioHen DESC
END

GO


# [dbo].[sp_GetDanhSachVaccine]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_GetDanhSachVaccine]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- 8. SP Lấy danh sách Vaccine
CREATE   PROCEDURE [dbo].[sp_GetDanhSachVaccine]
AS
BEGIN
    SELECT MaVC, TenVC, LoaiVC, GiaVC 
    FROM VACCINE
    ORDER BY TenVC
END

GO


# [dbo].[sp_GetDoanhThu]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_GetDoanhThu]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- Lấy tổng doanh thu theo tháng / theo năm
-- Nếu input vào tháng cụ thể và năm cụ thể -> output doanh thu trong tháng của năm đó
-- Nếu input vào năm (bỏ qua tháng) -> output toàn bộ doanh thu của 12 tháng trong năm đó
CREATE   PROCEDURE [dbo].[sp_GetDoanhThu]
    @Thang INT = NULL, -- Nếu NULL thì lấy cả năm
    @Nam INT
AS
BEGIN
    SET NOCOUNT ON;

    -- Trường hợp 1: Lấy chi tiết từng tháng trong năm
    IF @Thang IS NULL
    BEGIN
        SELECT 
            MONTH(NgayLap) AS Thang,
            SUM(TongTien) AS DoanhThu
        FROM HOA_DON
        WHERE YEAR(NgayLap) = @Nam
        GROUP BY MONTH(NgayLap)
        ORDER BY Thang ASC;
    END
    -- Trường hợp 2: Lấy tổng doanh thu của một tháng cụ thể
    ELSE
    BEGIN
        SELECT 
            ISNULL(SUM(TongTien), 0) AS DoanhThu
        FROM HOA_DON
        WHERE MONTH(NgayLap) = @Thang AND YEAR(NgayLap) = @Nam;
    END
END
GO


# [dbo].[sp_GetDoanhThu_NangCao]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_GetDoanhThu_NangCao]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE   PROCEDURE [dbo].[sp_GetDoanhThu_NangCao]
    @MaCN VARCHAR(20) = NULL, -- NULL = Lấy tất cả chi nhánh
    @TuNgay DATE = NULL,      -- NULL = Từ ngày đầu tiên
    @DenNgay DATE = NULL      -- NULL = Đến hiện tại
AS
BEGIN
    SET NOCOUNT ON;

    -- 1. Chuẩn bị biến để hiển thị ra cột (Xử lý trường hợp NULL)
    DECLARE @HienThiTuNgay DATE;
    DECLARE @HienThiDenNgay DATE;

    -- Nếu @TuNgay là NULL, lấy ngày của hóa đơn đầu tiên trong hệ thống
    IF @TuNgay IS NULL
        SELECT @HienThiTuNgay = MIN(CAST(NgayLap AS DATE)) FROM HOA_DON;
    ELSE
        SET @HienThiTuNgay = @TuNgay;

    -- Nếu @DenNgay là NULL, lấy ngày hiện tại
    IF @DenNgay IS NULL
        SET @HienThiDenNgay = CAST(GETDATE() AS DATE);
    ELSE
        SET @HienThiDenNgay = @DenNgay;

    -- 2. Thực hiện truy vấn
    SELECT 
        cn.MaCN,
        cn.TenChiNhanh,
        
        -- Cột mới: Hiển thị khoảng thời gian báo cáo cho người đọc biết
        @HienThiTuNgay AS NgayBatDau,
        @HienThiDenNgay AS NgayKetThuc,

        -- Số liệu thống kê
        COUNT(hd.MaHD) AS SoLuongHoaDon, 
        ISNULL(SUM(hd.TongTien), 0) AS TongDoanhThu
    FROM CHI_NHANH cn
    LEFT JOIN HOA_DON hd ON cn.MaCN = hd.MaCN
    WHERE 
        -- Lọc Chi Nhánh
        (@MaCN IS NULL OR cn.MaCN = @MaCN)
        
        -- Lọc Ngày (Sử dụng tham số gốc để lọc đúng logic)
        AND (@TuNgay IS NULL OR CAST(hd.NgayLap AS DATE) >= @TuNgay)
        AND (@DenNgay IS NULL OR CAST(hd.NgayLap AS DATE) <= @DenNgay)
    GROUP BY cn.MaCN, cn.TenChiNhanh
    ORDER BY TongDoanhThu DESC;
END
GO


# [dbo].[sp_GetDoanhThu_SanPham]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_GetDoanhThu_SanPham]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE   PROCEDURE [dbo].[sp_GetDoanhThu_SanPham]
    @TuNgay DATE = NULL,
    @DenNgay DATE = NULL,
    @MaCN VARCHAR(20) = NULL
AS
BEGIN
    SET NOCOUNT ON;
    DECLARE @Start DATE = ISNULL(@TuNgay, '2000-01-01');
    DECLARE @End DATE = ISNULL(@DenNgay, GETDATE());

    SELECT 
        sp.MaSP,
        sp.TenSP,
        sp.LoaiSP,
        SUM(ctmh.SoLuong) AS SoLuongBan,
        -- Giả sử giá bán nằm trong bảng SAN_PHAM hoặc tính từ HOA_DON. 
        -- Ở đây tạm tính tổng tiền hóa đơn của giao dịch mua hàng này
        SUM(hd.TongTien) AS DoanhThuUocTinh
    FROM SAN_PHAM sp
    JOIN CHI_TIET_MUA_HANG ctmh ON sp.MaSP = ctmh.MaSP
    JOIN DV_MUA_HANG dmh ON ctmh.MaMuaHang = dmh.MaMuaHang
    JOIN DICH_VU dv ON dmh.MaMuaHang = dv.MaDV -- Mua hàng cũng là 1 dịch vụ
    JOIN CHI_TIET_DV_SD ctsd ON dv.MaDV = ctsd.MaDV
    JOIN HOA_DON hd ON ctsd.MaHD = hd.MaHD
    WHERE 
        CAST(hd.NgayLap AS DATE) BETWEEN @Start AND @End
        AND (@MaCN IS NULL OR dv.MaCN = @MaCN)
    GROUP BY sp.MaSP, sp.TenSP, sp.LoaiSP
    ORDER BY SoLuongBan DESC;
END

GO


# [dbo].[sp_GetDoanhThu_TheoBacSi]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_GetDoanhThu_TheoBacSi]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE   PROCEDURE [dbo].[sp_GetDoanhThu_TheoBacSi]
    @TuNgay DATE = NULL,
    @DenNgay DATE = NULL,
    @MaCN VARCHAR(20) = NULL
AS
BEGIN
    SET NOCOUNT ON;

    -- Xử lý ngày null
    DECLARE @Start DATE = ISNULL(@TuNgay, '2000-01-01');
    DECLARE @End DATE = ISNULL(@DenNgay, GETDATE());

    SELECT 
        nv.MaNV,
        nv.HoTen AS TenBacSi,
        cn.TenChiNhanh,
        COUNT(DISTINCT dk.MaKham) AS SoLuotKham,
        ISNULL(SUM(hd.TongTien), 0) AS TongDoanhThu
    FROM NHAN_VIEN nv
    JOIN DV_KHAM dk ON nv.MaNV = dk.BacSiPhuTrach
    JOIN DICH_VU dv ON dk.MaKham = dv.MaDV
    JOIN CHI_NHANH cn ON dv.MaCN = cn.MaCN
    JOIN CHI_TIET_DV_SD ctsd ON dv.MaDV = ctsd.MaDV
    JOIN HOA_DON hd ON ctsd.MaHD = hd.MaHD
    WHERE 
        CAST(hd.NgayLap AS DATE) BETWEEN @Start AND @End
        AND (@MaCN IS NULL OR cn.MaCN = @MaCN)
    GROUP BY nv.MaNV, nv.HoTen, cn.TenChiNhanh
    ORDER BY TongDoanhThu DESC;
END

GO


# [dbo].[sp_GetKhachHangInfoBySdt]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_GetKhachHangInfoBySdt]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- 4. Tìm khách hàng theo SĐT (Chỉ lấy Mã và Tên để hiển thị và gán biến)
CREATE   PROCEDURE [dbo].[sp_GetKhachHangInfoBySdt]
    @SoDt VARCHAR(20)
AS
BEGIN
	IF @SoDt IS NULL
	BEGIN
		PRINT N'Số điện thoại không được để trống';
		RETURN;
	END

    SELECT MaKH, HoTen 
    FROM KHACH_HANG 
    WHERE SoDt = @SoDt
END

GO


# [dbo].[sp_GetLichHenChoDuyet]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_GetLichHenChoDuyet]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- 1. Lấy danh sách lịch hẹn (Chỉ lấy các cột hiển thị lên Grid)
CREATE   PROCEDURE [dbo].[sp_GetLichHenChoDuyet]
AS
BEGIN
    SELECT 
        l.MaLichHen,
        ISNULL(k.HoTen, N'Khách vãng lai') AS TenKhachHang, -- Xử lý null ngay tại SQL
        ISNULL(k.SoDt, '') AS SoDT,
        l.NgayHen,
        l.GioHen,
        ISNULL(nv.HoTen, '') AS TenBacSi,
        ISNULL(cn.TenChiNhanh, '') AS TenChiNhanh,
        l.TrangThai,
        l.GhiChu
    FROM LICH_HEN l
    JOIN KHACH_HANG k ON l.MaKH = k.MaKH
    JOIN NHAN_VIEN nv ON l.MaBS = nv.MaNV
    JOIN CHI_NHANH cn ON l.MaCN = cn.MaCN
    WHERE l.TrangThai = 'ChoXacNhan' OR l.TrangThai IS NULL OR l.TrangThai = ''
END

GO


# [dbo].[sp_GetLichSuKham]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_GetLichSuKham]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- 7. Lấy danh sách lịch sử khám thông qua mã thú cưng
CREATE   PROCEDURE [dbo].[sp_GetLichSuKham]
    @MaTC VARCHAR(20)
AS
BEGIN
    SET NOCOUNT ON;

	IF @MaTC IS NULL
	BEGIN
		;THROW 50001, 'MaTC không được để trống', 1;
	END

    SELECT 
        hd.NgayLap AS NgayKham,         -- Ngày khách đến khám
        dk.TrieuChung,
        dk.ChuanDoan,
        dk.ToaThuoc,
        dk.NgayTaiKham,                 -- Ngày hẹn tái khám
        nv.HoTen AS BacSiPhuTrach,      -- Tên bác sĩ
        dv.MaDV AS MaHoSo               -- Mã dịch vụ để tiện tra cứu sâu hơn nếu cần
    FROM DV_KHAM dk
    JOIN DICH_VU dv ON dk.MaKham = dv.MaDV
    JOIN CHI_TIET_DV_SD ctsd ON dv.MaDV = ctsd.MaDV
    JOIN HOA_DON hd ON ctsd.MaHD = hd.MaHD
    JOIN NHAN_VIEN nv ON dk.BacSiPhuTrach = nv.MaNV
    WHERE dv.MaTC = @MaTC
    ORDER BY hd.NgayLap DESC;
END

GO


# [dbo].[sp_GetNhanVien_ID]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_GetNhanVien_ID]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE   PROCEDURE [dbo].[sp_GetNhanVien_ID]
    @username NVARCHAR(10)
AS
BEGIN
    SET NOCOUNT ON;

    SELECT 
        MaNV,
        HoTen,
        ChucVu
    FROM 
        NHAN_VIEN
    WHERE 
        UserName = @username;
END
GO


# [dbo].[sp_GetSoLuotKham]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_GetSoLuotKham]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- Thống kê tổng số lượt khám từ ngày đến ngày
CREATE   PROCEDURE [dbo].[sp_GetSoLuotKham]
    @TuNgay DATE = NULL,
    @DenNgay DATE = NULL,
    @MaCN VARCHAR(20) = NULL
AS
BEGIN
    SET NOCOUNT ON;

    DECLARE @TuNgay_Thuc DATE;
    DECLARE @DenNgay_Thuc DATE;

    -- Lấy từ hóa đơn đầu tới hóa đơn cuối nếu người dùng để null ngày
    SELECT
        @TuNgay_Thuc = ISNULL(@TuNgay, MIN(CAST(hd.NgayLap AS DATE))),
        @DenNgay_Thuc = ISNULL(@DenNgay, MAX(CAST(hd.NgayLap AS DATE)))
    FROM HOA_DON hd;
    SELECT
        CAST(dv.MaCN AS VARCHAR(20)) AS MaChiNhanh,
        @TuNgay_Thuc AS TuNgay,
        @DenNgay_Thuc AS DenNgay,
        COUNT(dk.MaKham) AS TongSoLuotKham
    FROM DV_KHAM dk
    JOIN DICH_VU dv ON dk.MaKham = dv.MaDV
    JOIN CHI_TIET_DV_SD ctsd ON dv.MaDV = ctsd.MaDV
    JOIN HOA_DON hd ON ctsd.MaHD = hd.MaHD
    WHERE 
        CAST(hd.NgayLap AS DATE) BETWEEN @TuNgay_Thuc AND @DenNgay_Thuc
        AND (@MaCN IS NULL OR dv.MaCN = @MaCN)
    GROUP BY CAST(dv.MaCN AS VARCHAR(20));
END

GO


# [dbo].[sp_GetSoLuotKham_DateFilter]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_GetSoLuotKham_DateFilter]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
-- Thống kê số lượt khám từng ngày, từ ngày nào đến ngày nào
CREATE   PROCEDURE [dbo].[sp_GetSoLuotKham_DateFilter]
    @TuNgay DATE = NULL,
    @DenNgay DATE = NULL,
    @MaCN VARCHAR(20) = NULL
AS
BEGIN
    SET NOCOUNT ON;

    SELECT 
		CAST(dv.MaCN AS VARCHAR(20)) AS MaChiNhanh,
        CAST(hd.NgayLap AS DATE) AS Ngay,
        COUNT(dk.MaKham) AS SoLuotKham
    FROM DV_KHAM dk
    JOIN DICH_VU dv ON dk.MaKham = dv.MaDV
    JOIN CHI_TIET_DV_SD ctsd ON dv.MaDV = ctsd.MaDV
    JOIN HOA_DON hd ON ctsd.MaHD = hd.MaHD
    WHERE 
        (@TuNgay IS NULL OR CAST(hd.NgayLap AS DATE) >= @TuNgay)
        AND (@DenNgay IS NULL OR CAST(hd.NgayLap AS DATE) <= @DenNgay)
        AND (@MaCN IS NULL OR dv.MaCN = @MaCN)
    GROUP BY CAST(dv.MaCN AS VARCHAR(20)), CAST(hd.NgayLap AS DATE)
    ORDER BY Ngay ASC;
END
GO


# [dbo].[sp_GetThongTinKhamBenh]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_GetThongTinKhamBenh]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- 9. SP Lấy thông tin chi tiết Lịch Hẹn để hiển thị lên Form
CREATE   PROCEDURE [dbo].[sp_GetThongTinKhamBenh]
    @MaLichHen VARCHAR(20)
AS
BEGIN
    SELECT 
        l.MaLichHen,
        l.MaKH,
        k.HoTen AS TenKhachHang,
        l.MaBS,
        bs.HoTen AS TenBacSi,
        l.NgayHen,
        l.MaCN,
        -- Lấy mã thú cưng đầu tiên của khách (nếu chưa có trong lịch hẹn)
        (SELECT TOP 1 MaTC FROM THU_CUNG WHERE MaKH = l.MaKH) AS MaThuCung
    FROM LICH_HEN l
    LEFT JOIN KHACH_HANG k ON l.MaKH = k.MaKH
    LEFT JOIN NHAN_VIEN bs ON l.MaBS = bs.MaNV
    WHERE l.MaLichHen = @MaLichHen
END

GO


# [dbo].[sp_GetThuCungInfoByMaKh]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_GetThuCungInfoByMaKh]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- 5. Lấy danh sách thú cưng của khách (Chỉ lấy Mã và Tên để đổ vào ComboBox)
CREATE   PROCEDURE [dbo].[sp_GetThuCungInfoByMaKh]
    @MaKh VARCHAR(20)
AS
BEGIN
	IF @MaKh IS NULL
	BEGIN
		PRINT N'Mã khách hàng bị thiếu';
		RETURN;
	END	

    SELECT MaTC, TenTC 
    FROM THU_CUNG 
    WHERE MaKH = @MaKh
END

GO


# [dbo].[sp_HoanTatKhamBenh]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_HoanTatKhamBenh]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- 11. SP Hoàn tất khám (Cập nhật lịch hẹn)
CREATE   PROCEDURE [dbo].[sp_HoanTatKhamBenh]
    @MaLichHen VARCHAR(20),
    @GhiChu NVARCHAR(200)
AS
BEGIN
    UPDATE LICH_HEN 
    SET TrangThai = 'DaHoanThanh',
        GhiChu = ISNULL(GhiChu, '') + ' | ' + @GhiChu
    WHERE MaLichHen = @MaLichHen
END

GO


# [dbo].[sp_KiemTraNhanVienTonTai]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_KiemTraNhanVienTonTai]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- Kiểm tra sinh viên tồn tại bằng tên hoặc bằng mã nhân viên
CREATE   PROCEDURE [dbo].[sp_KiemTraNhanVienTonTai]
    @MaNV varchar(20) = NULL,   
	@HoTen nvarchar(100) = NULL 
AS
BEGIN
    IF (@MaNV IS NULL OR @MaNV = '') AND (@HoTen IS NULL OR @HoTen = '')
    BEGIN
        SELECT 1;
        RETURN;
    END

    SELECT TOP 1 1 
    FROM NHAN_VIEN 
    WHERE 
        ((@MaNV IS NULL OR @MaNV = '') OR MaNV = @MaNV)
        AND
        ((@HoTen IS NULL OR @HoTen = '') OR HoTen LIKE N'%' + @HoTen + N'%')
END

GO


# [dbo].[sp_LayChiTietHoaDonDayDu]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_LayChiTietHoaDonDayDu]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- Stored Procedure 7: Lấy chi tiết hóa đơn đầy đủ (tối ưu)
CREATE PROCEDURE [dbo].[sp_LayChiTietHoaDonDayDu]
    @MaHD VARCHAR(20)
AS
BEGIN
    SET NOCOUNT ON;
    
    -- Thông tin hóa đơn và liên kết
    SELECT 
        h.MaHD,
        h.NgayLap,
        h.TrangThai,
        h.TongTien,
        h.KhuyenMai,
        h.HinhThucThanhToan,
        kh.HoTen AS TenKH,
        kh.SoDT,
        kh.CapHoiVien,
        nv.HoTen AS TenNV,
        cn.TenChiNhanh AS TenCN
    FROM HOA_DON h
    LEFT JOIN KHACH_HANG kh ON h.MaKH = kh.MaKH
    LEFT JOIN NHAN_VIEN nv ON h.MaNV = nv.MaNV
    LEFT JOIN CHI_NHANH cn ON h.MaCN = cn.MaCN
    WHERE h.MaHD = @MaHD;
    
    -- Danh sách sản phẩm
    SELECT 
        sp.TenSP,
        ctmh.SoLuong,
        ctmh.DonGia,
        (ctmh.SoLuong * ctmh.DonGia) AS ThanhTien
    FROM CHI_TIET_DV_SD ctdv
    INNER JOIN CHI_TIET_MUA_HANG ctmh ON ctdv.MaDV = ctmh.MaMuaHang
    INNER JOIN SAN_PHAM sp ON ctmh.MaSP = sp.MaSP
    WHERE ctdv.MaHD = @MaHD;
END

GO


# [dbo].[sp_LayDanhSachSanPhamTheoHoaDon]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_LayDanhSachSanPhamTheoHoaDon]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- Stored Procedure 5: Lấy danh sách sản phẩm theo hóa đơn
CREATE PROCEDURE [dbo].[sp_LayDanhSachSanPhamTheoHoaDon]
    @MaHD VARCHAR(20)
AS
BEGIN
    SET NOCOUNT ON;
    
    SELECT 
        sp.TenSP,
        ctmh.SoLuong,
        ctmh.DonGia,
        (ctmh.SoLuong * ctmh.DonGia) AS ThanhTien
    FROM CHI_TIET_DV_SD ctdv
    INNER JOIN CHI_TIET_MUA_HANG ctmh ON ctdv.MaDV = ctmh.MaMuaHang
    INNER JOIN SAN_PHAM sp ON ctmh.MaSP = sp.MaSP
    WHERE ctdv.MaHD = @MaHD;
END

GO


# [dbo].[sp_LayThongTinChiNhanh]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_LayThongTinChiNhanh]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- Stored Procedure 4: Lấy thông tin chi nhánh
CREATE PROCEDURE [dbo].[sp_LayThongTinChiNhanh]
    @MaCN VARCHAR(20)
AS
BEGIN
    SET NOCOUNT ON;
    
    SELECT 
        MaCN,
        TenChiNhanh
    FROM CHI_NHANH
    WHERE MaCN = @MaCN;
END

GO


# [dbo].[sp_LayThongTinGoiTiem]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_LayThongTinGoiTiem]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROC [dbo].[sp_LayThongTinGoiTiem]
    @MaTC varchar(20)
AS
BEGIN
    SET NOCOUNT ON;

    -- Kiểm tra có gói tiêm còn hiệu lực không
    IF NOT EXISTS (
        SELECT 1
        FROM DICH_VU dv
        JOIN DV_TIEM_PHONG_THEO_THANG gt ON dv.MaDV = gt.MaGoi
        WHERE dv.MaTC = @MaTC
          AND gt.NgayKT >= CAST(GETDATE() AS date)
    )
    BEGIN
        -- Không có gói tiêm
        SELECT 
            N'Thú cưng không có gói tiêm còn hiệu lực' AS ThongBao;
        RETURN;
    END

    -- Có gói tiêm thì trả chi tiết
    SELECT
        gt.MaGoi,
        gt.TenGoi,
        gt.SoThang,
        gt.NgayDK,
        gt.NgayKT,
        gt.GiaGoi,
        vc.MaVC,
        vc.TenVC,
        vc.LoaiVC,
        ctt.LieuLuong,
        ctt.LanTiem,
        ctt.NgayTiem
    FROM
		DICH_VU dv
		JOIN DV_TIEM_PHONG_THEO_THANG gt ON dv.MaDV = gt.MaGoi
		JOIN CHI_TIET_TIEM_THANG ctt ON gt.MaGoi = ctt.MaGoi
		JOIN VACCINE vc ON ctt.MaVC = vc.MaVC
    WHERE
		dv.MaTC = @MaTC
		AND gt.NgayKT >= CAST(GETDATE() AS date)
    ORDER BY ctt.LanTiem;
END

GO


# [dbo].[sp_LayThongTinKhachHang]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_LayThongTinKhachHang]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- Stored Procedure 2: Lấy thông tin khách hàng
CREATE PROCEDURE [dbo].[sp_LayThongTinKhachHang]
    @MaKH VARCHAR(20)
AS
BEGIN
    SET NOCOUNT ON;
    
    SELECT 
        MaKH,
        HoTen,
        SoDT,
        CapHoiVien
    FROM KHACH_HANG
    WHERE MaKH = @MaKH;
END

GO


# [dbo].[sp_LayThongTinNhanVien]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_LayThongTinNhanVien]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- Stored Procedure 3: Lấy thông tin nhân viên
CREATE PROCEDURE [dbo].[sp_LayThongTinNhanVien]
    @MaNV VARCHAR(20)
AS
BEGIN
    SET NOCOUNT ON;
    
    SELECT 
        MaNV,
        HoTen
    FROM NHAN_VIEN
    WHERE MaNV = @MaNV;
END

GO


# [dbo].[sp_LoadChiNhanh]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_LoadChiNhanh]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- 1. Load Chi Nhánh
--DROP PROCEDURE sp_LoadChiNhanh
--GO
CREATE PROCEDURE [dbo].[sp_LoadChiNhanh]
AS
BEGIN
    SELECT MaCN, TenChiNhanh, DiaChi, SoDienThoai, GioMoCua, GioDongCua
    FROM CHI_NHANH 
    ORDER BY MaCN
END
GO


# [dbo].[sp_LoadProductsByChiNhanh]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_LoadProductsByChiNhanh]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- 2. Load Sản Phẩm theo Chi Nhánh với Tồn Kho
--DROP PROCEDURE sp_LoadProductsByChiNhanh
--GO
CREATE PROCEDURE [dbo].[sp_LoadProductsByChiNhanh]
    @MaCN VARCHAR(20)
AS
BEGIN
    SELECT 
        sp.MaSP,
        sp.TenSP,
        ISNULL(sp.GiaBan, 0) AS GiaBan,
        sp.LoaiSP,
        ISNULL(SUM(tk.SoLuong), 0) AS TonKho,
        @MaCN AS MaCN
    FROM SAN_PHAM sp
    LEFT JOIN TON_KHO_SAN_PHAM tk ON sp.MaSP = tk.MaSP AND tk.MaCN = @MaCN
    GROUP BY sp.MaSP, sp.TenSP, sp.GiaBan, sp.LoaiSP
    HAVING ISNULL(SUM(tk.SoLuong), 0) > 0
    ORDER BY sp.TenSP
END
GO


# [dbo].[sp_LuuBenhAnKham]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_LuuBenhAnKham]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- 10. SP Lưu Bệnh Án (Thực hiện Transaction để đảm bảo tính toàn vẹn)
CREATE   PROCEDURE [dbo].[sp_LuuBenhAnKham]
    @MaDichVu VARCHAR(20),
    @MaThuCung VARCHAR(20),
    @MaChiNhanh VARCHAR(20),
    @TrieuChung NVARCHAR(100),
    @ChuanDoan NVARCHAR(100),
    @ToaThuoc NVARCHAR(100),
    @MaBacSi VARCHAR(20),
    @GiaKham FLOAT
AS
BEGIN
    BEGIN TRANSACTION;
    BEGIN TRY
        -- Insert DICH_VU
        INSERT INTO DICH_VU(MaDV, MaTC, MaCN)
        VALUES(@MaDichVu, @MaThuCung, @MaChiNhanh);

        -- Insert DV_KHAM
        INSERT INTO DV_KHAM(MaKham, TrieuChung, ChuanDoan, ToaThuoc, BacSiPhuTrach, GiaKhamBenh)
        VALUES(@MaDichVu, @TrieuChung, @ChuanDoan, @ToaThuoc, @MaBacSi, @GiaKham);

        COMMIT TRANSACTION;
    END TRY
    BEGIN CATCH
        ROLLBACK TRANSACTION;
        THROW;
    END CATCH
END

GO


# [dbo].[sp_PayInvoice]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_PayInvoice]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO


-- 8. Thanh Toán Hóa Đơn
--DROP PROCEDURE sp_PayInvoice
CREATE PROCEDURE [dbo].[sp_PayInvoice]
    @MaHD VARCHAR(20),
    @MaKH VARCHAR(20) = NULL
AS
BEGIN
    BEGIN TRANSACTION
    BEGIN TRY
        -- Kiểm tra hóa đơn tồn tại
        DECLARE @TongTien DECIMAL(18,0)
        DECLARE @TrangThai NVARCHAR(100)
        
        SELECT @TongTien = TongTien, @TrangThai = TrangThai
        FROM HOA_DON
        WHERE MaHD = @MaHD

        IF @TongTien IS NULL
        BEGIN
            THROW 50002, N'Không tìm thấy hóa đơn', 1
        END

        IF @TrangThai = N'DaThanhToan'
        BEGIN
            THROW 50003, N'Hóa đơn đã được thanh toán', 1
        END

        -- Cập nhật trạng thái hóa đơn
        UPDATE HOA_DON
        SET TrangThai = N'DaThanhToan'
        WHERE MaHD = @MaHD

        COMMIT TRANSACTION
    END TRY
    BEGIN CATCH
        ROLLBACK TRANSACTION
        THROW
    END CATCH
END
GO


# [dbo].[sp_SearchCustomer]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_SearchCustomer]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- 4. Tìm Khách Hàng theo SĐT hoặc Tên
--DROP PROCEDURE sp_SearchCustomer
--GO
CREATE PROCEDURE [dbo].[sp_SearchCustomer]
    @SearchText NVARCHAR(100)
AS
BEGIN
    SELECT 
        MaKH,
        HoTen,
        SoDT,
        Email,
        GioiTinh,
        CapHoiVien,
        ISNULL(DiemTichLuy, 0) AS DiemTichLuy
    FROM KHACH_HANG
    WHERE SoDT = @SearchText OR HoTen LIKE '%' + @SearchText + '%'
END
GO


# [dbo].[sp_SearchEmployee]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_SearchEmployee]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- 3. Tìm Nhân Viên theo Mã hoặc Tên và Chi Nhánh
--DROP PROCEDURE sp_SearchEmployee
--GO
CREATE PROCEDURE [dbo].[sp_SearchEmployee]
    @SearchText NVARCHAR(100),
    @MaCN VARCHAR(20)
AS
BEGIN
    SELECT 
        nv.MaNV,
        nv.HoTen,
        nv.MaCN,
        cn.TenChiNhanh
    FROM NHAN_VIEN nv
    LEFT JOIN CHI_NHANH cn ON nv.MaCN = cn.MaCN
    WHERE (nv.MaNV = @SearchText OR nv.HoTen LIKE '%' + @SearchText + '%')
    AND nv.MaCN = @MaCN
END
GO


# [dbo].[sp_ThemCaLamViec]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_ThemCaLamViec]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- Them ca lam viec
CREATE   PROC [dbo].[sp_ThemCaLamViec]
    @MaCa VARCHAR(20),
    @MaNV VARCHAR(20),
    @NgayLamViec DATE
AS
BEGIN
    SET NOCOUNT ON;

    -- 1. Kiem tra NV co ton tai
    IF NOT EXISTS (SELECT 1 FROM NHAN_VIEN WHERE MaNV = @MaNV)
    BEGIN
        ;THROW 52001, N'Nhân viên không tồn tại.', 1;
        RETURN;
    END

    -- 2. Kiem tra ca lam viec co ton tai
    IF NOT EXISTS (SELECT 1 FROM CA_LAM_VIEC WHERE MaCa = @MaCa)
    BEGIN
        ;THROW 52002, N'Ca làm việc không tồn tại.', 1;
        RETURN;
    END

    -- 3. Kiem tra nhan vien da duoc phan cong vao ca do trong ngay hom do chua
    IF EXISTS (SELECT 1 FROM BANG_PHAN_CA WHERE MaCa = @MaCa AND MaNV = @MaNV AND NgayLamViec = @NgayLamViec)
    BEGIN
        ;THROW 52003, N'Nhân viên đã được phân công ca này trong ngày đã chọn.', 1;
        RETURN;
    END

    -- 4. Them vao ca moi
    INSERT INTO BANG_PHAN_CA (MaCa, MaNV, NgayLamViec)
    VALUES (@MaCa, @MaNV, @NgayLamViec);
END

GO


# [dbo].[sp_ThemPhanCa]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_ThemPhanCa]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

--Thêm phân ca
CREATE   PROCEDURE [dbo].[sp_ThemPhanCa]
    @TenCa nvarchar(50),          -- Tên ca
    @InputNhanVien nvarchar(100), -- MaNV hoặc HoTen
    @NgayLamViec date            
AS
BEGIN
    SET NOCOUNT ON;

    DECLARE @MaCa varchar(20);
    DECLARE @MaNV varchar(20);

	-- 1. TenCa -> MaCa
    SELECT TOP 1 @MaCa = MaCa 
    FROM CA_LAM_VIEC 
    WHERE TenCa = @TenCa;

    -- Validation: Không tìm thấy mã ca
    IF @MaCa IS NULL
    BEGIN
        ;THROW 51000, N'Lỗi: Không tìm thấy Tên Ca làm việc này trong hệ thống.', 1;
        RETURN;
    END

    -- 2. Input -> MaNV
    SELECT TOP 1 @MaNV = MaNV 
    FROM NHAN_VIEN 
    WHERE MaNV = @InputNhanVien OR HoTen = @InputNhanVien;

    -- Không tìm thấy nhân viên
    IF @MaNV IS NULL
    BEGIN
        ;THROW 51000, N'Lỗi: Không tìm thấy Nhân viên (Vui lòng kiểm tra lại Mã hoặc Họ Tên).', 1;
        RETURN;
    END

    -- 3. Kiểm tra NV đã được phân công vào thời gian đó chứa
    IF EXISTS (SELECT 1 FROM BANG_PHAN_CA 
               WHERE MaNV = @MaNV AND MaCa = @MaCa AND NgayLamViec = @NgayLamViec)
    BEGIN
        DECLARE @ErrorMsg nvarchar(200);
        SET @ErrorMsg = N'Lỗi: Nhân viên ' + @InputNhanVien + N' đã có lịch làm việc này rồi.';
        ;THROW 51000, @ErrorMsg, 1;
        RETURN;
    END

    -- 4. Thêm dữ liệu
    INSERT INTO BANG_PHAN_CA (MaCa, MaNV, NgayLamViec)
    VALUES (@MaCa, @MaNV, @NgayLamViec);
END

GO


# [dbo].[sp_ThongKe_DoanhThuPhongKham]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_ThongKe_DoanhThuPhongKham]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- =============================================
-- 1. Thống kê tổng doanh thu (Danh sách hóa đơn chi tiết)
-- =============================================
CREATE   PROCEDURE [dbo].[sp_ThongKe_DoanhThuPhongKham]
    @TuNgay DATE = NULL,
    @DenNgay DATE = NULL,
    @MaCN VARCHAR(20) = NULL,
    @TuKhoa NVARCHAR(100) = NULL, -- Tìm theo tên KH hoặc Mã HĐ
    @SortOption INT = 0 -- 0: Mới nhất, 1: Cũ nhất, 2: Giá cao
AS
BEGIN
    SET NOCOUNT ON;

    SELECT 
        hd.MaHD,
        hd.NgayLap,
        hd.TongTien,
        hd.HinhThucThanhToan,
        kh.HoTen AS TenKhachHang,
        nv.HoTen AS NhanVienTao,
        cn.TenChiNhanh
    FROM HOA_DON hd
    JOIN KHACH_HANG kh ON hd.MaKH = kh.MaKH
    JOIN NHAN_VIEN nv ON hd.MaNV = nv.MaNV
    JOIN CHI_NHANH cn ON hd.MaCN = cn.MaCN
    WHERE
        (@TuNgay IS NULL OR CAST(hd.NgayLap AS DATE) >= @TuNgay)
        AND (@DenNgay IS NULL OR CAST(hd.NgayLap AS DATE) <= @DenNgay)
        AND (@MaCN IS NULL OR hd.MaCN = @MaCN)
        AND (@TuKhoa IS NULL OR kh.HoTen LIKE N'%' + @TuKhoa + '%' OR hd.MaHD LIKE N'%' + @TuKhoa + '%')
    ORDER BY 
        CASE WHEN @SortOption = 0 THEN hd.NgayLap END DESC,
        CASE WHEN @SortOption = 1 THEN hd.NgayLap END ASC,
        CASE WHEN @SortOption = 2 THEN hd.TongTien END DESC;
END

GO


# [dbo].[sp_ThongKe_DoanhThuSanPham]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_ThongKe_DoanhThuSanPham]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- =============================================
-- 4. Thống kê doanh thu theo Sản Phẩm
-- =============================================
CREATE   PROCEDURE [dbo].[sp_ThongKe_DoanhThuSanPham]
    @TuNgay DATE = NULL,
    @DenNgay DATE = NULL,
    @MaCN VARCHAR(20) = NULL
AS
BEGIN
    SET NOCOUNT ON;

    SELECT TOP 20 -- Lấy top 20 sản phẩm
        sp.MaSP,
        sp.TenSP,
        sp.LoaiSP,
        SUM(ctmh.SoLuong) AS SoLuongBan,
        SUM(ctmh.SoLuong * ctmh.DonGia) AS DoanhThu
    FROM CHI_TIET_MUA_HANG ctmh
    JOIN SAN_PHAM sp ON ctmh.MaSP = sp.MaSP
    JOIN DV_MUA_HANG dvmh ON ctmh.MaMuaHang = dvmh.MaMuaHang
    JOIN DICH_VU dv ON dvmh.MaMuaHang = dv.MaDV
    JOIN CHI_TIET_DV_SD ctsd ON dv.MaDV = ctsd.MaDV
    JOIN HOA_DON hd ON ctsd.MaHD = hd.MaHD
    WHERE 
        (@TuNgay IS NULL OR CAST(hd.NgayLap AS DATE) >= @TuNgay)
        AND (@DenNgay IS NULL OR CAST(hd.NgayLap AS DATE) <= @DenNgay)
        AND (@MaCN IS NULL OR dv.MaCN = @MaCN)
    GROUP BY sp.MaSP, sp.TenSP, sp.LoaiSP
    ORDER BY DoanhThu DESC;
END

GO


# [dbo].[sp_ThongKe_DoanhThuTatCaChiNhanh]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_ThongKe_DoanhThuTatCaChiNhanh]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- =============================================
-- 5. Thống kê doanh thu tất cả chi nhánh
-- =============================================
CREATE   PROCEDURE [dbo].[sp_ThongKe_DoanhThuTatCaChiNhanh]
    @TuNgay DATE = NULL,
    @DenNgay DATE = NULL
AS
BEGIN
    SET NOCOUNT ON;

    SELECT 
        cn.MaCN,
        cn.TenChiNhanh,
        COUNT(hd.MaHD) AS SoLuongHoaDon,
        SUM(ISNULL(hd.TongTien, 0)) AS TongDoanhThu
    FROM CHI_NHANH cn
    LEFT JOIN HOA_DON hd ON cn.MaCN = hd.MaCN 
        AND (@TuNgay IS NULL OR CAST(hd.NgayLap AS DATE) >= @TuNgay)
        AND (@DenNgay IS NULL OR CAST(hd.NgayLap AS DATE) <= @DenNgay)
    GROUP BY cn.MaCN, cn.TenChiNhanh
    ORDER BY TongDoanhThu DESC;
END

GO


# [dbo].[sp_ThongKe_DoanhThuTheoBacSi]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_ThongKe_DoanhThuTheoBacSi]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- =============================================
-- 2. Thống kê doanh thu theo Bác Sĩ
-- =============================================
CREATE   PROCEDURE [dbo].[sp_ThongKe_DoanhThuTheoBacSi]
    @TuNgay DATE = NULL,
    @DenNgay DATE = NULL,
    @MaCN VARCHAR(20) = NULL
AS
BEGIN
    SET NOCOUNT ON;

    -- CTE lấy doanh thu từ Khám bệnh
    WITH DoanhThuKham AS (
        SELECT dk.BacSiPhuTrach AS MaNV, SUM(ISNULL(dk.GiaKhamBenh, 0)) AS TienKham
        FROM DV_KHAM dk
        JOIN DICH_VU dv ON dk.MaKham = dv.MaDV
        JOIN CHI_TIET_DV_SD ctsd ON dv.MaDV = ctsd.MaDV
        JOIN HOA_DON hd ON ctsd.MaHD = hd.MaHD
        WHERE 
            (@TuNgay IS NULL OR CAST(hd.NgayLap AS DATE) >= @TuNgay)
            AND (@DenNgay IS NULL OR CAST(hd.NgayLap AS DATE) <= @DenNgay)
            AND (@MaCN IS NULL OR dv.MaCN = @MaCN)
        GROUP BY dk.BacSiPhuTrach
    ),

    -- CTE lấy doanh thu từ Tiêm phòng đơn lẻ (Tính tổng giá vaccine tiêm)
    DoanhThuTiemLe AS (
        SELECT tp.BacSiPhuTrach AS MaNV, SUM(ISNULL(ctt.Gia, 0)) AS TienTiem
        FROM DV_TIEM_PHONG_DON_LE tp
        JOIN CHI_TIET_TIEM ctt ON tp.MaTiem = ctt.MaTiem
        JOIN DICH_VU dv ON tp.MaTiem = dv.MaDV
        JOIN CHI_TIET_DV_SD ctsd ON dv.MaDV = ctsd.MaDV
        JOIN HOA_DON hd ON ctsd.MaHD = hd.MaHD
        WHERE 
            (@TuNgay IS NULL OR CAST(hd.NgayLap AS DATE) >= @TuNgay)
            AND (@DenNgay IS NULL OR CAST(hd.NgayLap AS DATE) <= @DenNgay)
            AND (@MaCN IS NULL OR dv.MaCN = @MaCN)
        GROUP BY tp.BacSiPhuTrach
    )
    
    -- Tổng hợp kết quả
    SELECT 
        nv.MaNV,
        nv.HoTen AS TenBacSi,
        ISNULL(k.TienKham, 0) + ISNULL(t.TienTiem, 0) AS TongDoanhThu
    FROM NHAN_VIEN nv
    LEFT JOIN DoanhThuKham k ON nv.MaNV = k.MaNV
    LEFT JOIN DoanhThuTiemLe t ON nv.MaNV = t.MaNV
    WHERE nv.ChucVu LIKE 'BacSi'
          AND (ISNULL(k.TienKham, 0) + ISNULL(t.TienTiem, 0)) > 0 -- Chỉ hiện BS có doanh thu
    ORDER BY TongDoanhThu DESC;
END

GO


# [dbo].[sp_ThongKe_LuotKhamTheoChiNhanh]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_ThongKe_LuotKhamTheoChiNhanh]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- =============================================
-- 6. Thống kê lượt khám theo chi nhánh
-- =============================================
CREATE   PROCEDURE [dbo].[sp_ThongKe_LuotKhamTheoChiNhanh]
    @TuNgay DATE = NULL,
    @DenNgay DATE = NULL
AS
BEGIN
    SET NOCOUNT ON;

    SELECT 
        cn.MaCN,
        cn.TenChiNhanh,
        COUNT(dk.MaKham) AS SoLuotKham
    FROM CHI_NHANH cn
    LEFT JOIN DICH_VU dv ON cn.MaCN = dv.MaCN
    LEFT JOIN DV_KHAM dk ON dv.MaDV = dk.MaKham
    LEFT JOIN CHI_TIET_DV_SD ctsd ON dv.MaDV = ctsd.MaDV
    LEFT JOIN HOA_DON hd ON ctsd.MaHD = hd.MaHD
    WHERE 
        (@TuNgay IS NULL OR CAST(hd.NgayLap AS DATE) >= @TuNgay)
        AND (@DenNgay IS NULL OR CAST(hd.NgayLap AS DATE) <= @DenNgay)
    GROUP BY cn.MaCN, cn.TenChiNhanh
    ORDER BY SoLuotKham DESC;
END

GO


# [dbo].[sp_ThongKe_SoLuotKham]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_ThongKe_SoLuotKham]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- =============================================
-- 3. Thống kê số lượt khám bệnh
-- =============================================
CREATE   PROCEDURE [dbo].[sp_ThongKe_SoLuotKham]
    @TuNgay DATE = NULL,
    @DenNgay DATE = NULL,
    @MaCN VARCHAR(20)
AS
BEGIN
    SET NOCOUNT ON;
	
	IF @MaCN IS NULL
	BEGIN
		;THROW 50001, N'Mã chi nhánh không được rỗng', 1;
	END
    SELECT 
        CAST(hd.NgayLap AS DATE) AS Ngay,
        COUNT(dk.MaKham) AS SoLuotKham
    FROM DV_KHAM dk
    JOIN DICH_VU dv ON dk.MaKham = dv.MaDV
    JOIN CHI_TIET_DV_SD ctsd ON dv.MaDV = ctsd.MaDV
    JOIN HOA_DON hd ON ctsd.MaHD = hd.MaHD
    WHERE 
        (@TuNgay IS NULL OR CAST(hd.NgayLap AS DATE) >= @TuNgay)
        AND (@DenNgay IS NULL OR CAST(hd.NgayLap AS DATE) <= @DenNgay)
        AND (@MaCN IS NULL OR dv.MaCN = @MaCN)
    GROUP BY CAST(hd.NgayLap AS DATE)
    ORDER BY Ngay ASC;
END

GO


# [dbo].[sp_TimKiemCaLamViec]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_TimKiemCaLamViec]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- Tìm kiếm ca làm việc
CREATE   PROCEDURE [dbo].[sp_TimKiemCaLamViec]
    @MaNV varchar(20) = NULL,
    @HoTen nvarchar(100) = NULL,
    @Flag int -- 1 for ID, 0 for Name
AS
BEGIN
    SELECT 
        NV.MaNV, 
        NV.HoTen, 
        C.TenCa, 
        C.GioBD, 
        C.GioKT, 
        PC.NgayLamViec
    FROM BANG_PHAN_CA PC
    JOIN NHAN_VIEN NV ON PC.MaNV = NV.MaNV
    JOIN CA_LAM_VIEC C ON PC.MaCa = C.MaCa
    WHERE 
        ((@MaNV IS NULL OR @MaNV = '') AND (@HoTen IS NULL OR @HoTen = ''))       
        OR
        (@Flag = 1 AND NV.MaNV = @MaNV)
        OR 
        (@Flag = 0 AND NV.HoTen LIKE N'%' + @HoTen + N'%');
END

GO


# [dbo].[sp_TimKiemCaLamViec_TheoMaNV]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_TimKiemCaLamViec_TheoMaNV]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- Tìm kiếm ca làm việc theo mã nhân viên
CREATE   PROCEDURE [dbo].[sp_TimKiemCaLamViec_TheoMaNV]
    @MaNV VARCHAR(20)   -- Chắc chắn không NULL
AS
BEGIN
    SELECT 
        NV.MaNV,
        NV.HoTen,
        C.TenCa,
        C.GioBD,
        C.GioKT,
        PC.NgayLamViec
    FROM BANG_PHAN_CA PC
    JOIN NHAN_VIEN NV ON PC.MaNV = NV.MaNV
    JOIN CA_LAM_VIEC C ON PC.MaCa = C.MaCa
    WHERE NV.MaNV = @MaNV;
END

GO


# [dbo].[sp_TimKiemHoaDonChuaThanhToan]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_TimKiemHoaDonChuaThanhToan]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- Stored Procedure 1: Tìm kiếm hóa đơn chưa thanh toán
CREATE PROCEDURE [dbo].[sp_TimKiemHoaDonChuaThanhToan]
    @CustomerSearch NVARCHAR(100)
AS
BEGIN
    SET NOCOUNT ON;
    
    SELECT TOP 50
        h.MaHD,
        h.NgayLap,
        h.TrangThai,
        h.TongTien,
        h.KhuyenMai,
        h.HinhThucThanhToan,
        h.MaKH,
        h.MaNV,
        h.MaCN
    FROM HOA_DON h
    INNER JOIN KHACH_HANG kh ON h.MaKH = kh.MaKH
    WHERE h.TrangThai = 'ChuaThanhToan'
        AND (kh.SoDT LIKE '%' + @CustomerSearch + '%' 
             OR kh.HoTen LIKE '%' + @CustomerSearch + '%')
    ORDER BY h.NgayLap DESC;
END

GO


# [dbo].[sp_TinhLuong]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_TinhLuong]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- Tính lương tháng
CREATE   PROCEDURE [dbo].[sp_TinhLuong]
    @Thang INT,
    @Nam INT,
    @PhanTramHoaHong FLOAT = 0.02 -- Ví dụ: Hoa hồng 2% doanh số
AS
BEGIN
    SET NOCOUNT ON;

    -- 1. Xóa dữ liệu lương cũ của tháng đó (để tính lại từ đầu nếu chạy lại)
    DELETE FROM BANG_LUONG_THANG WHERE Thang = @Thang AND Nam = @Nam;

    -- 2. Tính toán và Insert vào bảng lương
    INSERT INTO BANG_LUONG_THANG (MaNV, Thang, Nam, TongGioCong, LuongCoBan, Thuong, TongLuong, NgayTinhLuong)
    SELECT 
        nv.MaNV,
        @Thang,
        @Nam,
        
        -- A. Tính Tổng Giờ Công (Từ bảng CHAM_CONG)
        ISNULL((
            SELECT SUM(DATEDIFF(MINUTE, cc.Checkin, cc.Checkout)) / 60.0
            FROM CHAM_CONG cc 
            WHERE cc.MaNV = nv.MaNV 
              AND MONTH(cc.NgayLamViec) = @Thang 
              AND YEAR(cc.NgayLamViec) = @Nam
        ), 0) AS TongGioCong,

        nv.LuongCoBan, -- Giả sử LuongCoBan ở đây là Lương theo giờ

        -- B. Tính Thưởng (Hoa hồng từ Hóa Đơn)
        ISNULL((
            SELECT SUM(hd.TongTien) * @PhanTramHoaHong
            FROM HOA_DON hd
            WHERE hd.MaNV = nv.MaNV 
              AND MONTH(hd.NgayLap) = @Thang 
              AND YEAR(hd.NgayLap) = @Nam
        ), 0) AS Thuong,

        -- C. Tính Tổng Lương = (Giờ * Lương Cơ Bản) + Thưởng
        0, -- Tạm thời để 0, sẽ update ở bước sau hoặc tính trực tiếp ở đây
        
        GETDATE() -- Ngày tính lương
    FROM NHAN_VIEN nv;

    -- 3. Cập nhật cột TongLuong (Tính toán cột cuối cùng)
    UPDATE BANG_LUONG_THANG
    SET TongLuong = (TongGioCong * LuongCoBan) + Thuong
    WHERE Thang = @Thang AND Nam = @Nam;
END

GO


# [dbo].[sp_XemBangLuong]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_XemBangLuong]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- Xem bảng lương để đưa ra quyết định trước khi thực hiện tính
CREATE   PROCEDURE [dbo].[sp_XemBangLuong]
    @Thang INT,
    @Nam INT
AS
BEGIN
    SET NOCOUNT ON;

    SELECT 
        bl.MaNV,
        nv.HoTen,
        nv.ChucVu,
        bl.TongGioCong,
        bl.LuongCoBan,
        bl.Thuong,
        bl.TongLuong
    FROM BANG_LUONG_THANG bl
    JOIN NHAN_VIEN nv ON bl.MaNV = nv.MaNV
    WHERE bl.Thang = @Thang AND bl.Nam = @Nam
    ORDER BY bl.TongLuong DESC;
END

GO


# [dbo].[sp_XoaCaLamViecc]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_XoaCaLamViecc]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- Xoa ca lam viec
CREATE   PROC [dbo].[sp_XoaCaLamViecc]
    @MaCa VARCHAR(20),
    @MaNV VARCHAR(20),
    @NgayLamViec DATE
AS
BEGIN
    SET NOCOUNT ON;

    IF NOT EXISTS (
        SELECT 1
        FROM BANG_PHAN_CA
        WHERE MaCa = @MaCa
          AND MaNV = @MaNV
          AND NgayLamViec = @NgayLamViec
    )
    BEGIN
        ;THROW 52004, N'Không tồn tại ca làm việc để xóa.', 2;
        RETURN;
    END

    DELETE FROM BANG_PHAN_CA
    WHERE MaCa = @MaCa
      AND MaNV = @MaNV
      AND NgayLamViec = @NgayLamViec;
END

GO


# [dbo].[sp_XoaPhanCa]

In [0]:
/****** Object:  StoredProcedure [dbo].[sp_XoaPhanCa]    Script Date: 2026-01-07 6:05:41 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- Xóa phân ca
CREATE   PROCEDURE [dbo].[sp_XoaPhanCa]
    @TenCa nvarchar(50),          
	@InputNhanVien nvarchar(100), -- MaNV hoặc HoTen
    @NgayLamViec date             
AS
BEGIN
    SET NOCOUNT ON;
	DECLARE @MaCa varchar(20);
    DECLARE @MaNV varchar(20);

    -- 1. (TenCa -> MaCa)
    SELECT TOP 1 @MaCa = MaCa 
    FROM CA_LAM_VIEC 
    WHERE TenCa = @TenCa;

    IF @MaCa IS NULL
    BEGIN
        ;THROW 51000, N'Lỗi: Không tìm thấy Tên Ca làm việc này.', 1;
        RETURN;
    END

    -- 2. (Input -> MaNV)
    SELECT TOP 1 @MaNV = MaNV 
    FROM NHAN_VIEN 
    WHERE MaNV = @InputNhanVien OR HoTen = @InputNhanVien;

    IF @MaNV IS NULL
    BEGIN
        ;THROW 51000, N'Lỗi: Không tìm thấy Nhân viên (Kiểm tra lại Mã hoặc Tên).', 1;
        RETURN;
    END

    -- 3. Kiểm tra mã nhân viên có trong bảng phân ca không
    IF NOT EXISTS (SELECT 1 FROM BANG_PHAN_CA 
                   WHERE MaNV = @MaNV AND MaCa = @MaCa AND NgayLamViec = @NgayLamViec)
    BEGIN
        ;THROW 51000, N'Lỗi: Không tìm thấy lịch làm việc này để xóa.', 1;
        RETURN;
    END

    -- 4. Xóa dữ liệu
    DELETE FROM BANG_PHAN_CA 
    WHERE MaNV = @MaNV AND MaCa = @MaCa AND NgayLamViec = @NgayLamViec;

    SELECT N'Xóa phân ca thành công!' AS ThongBao;
END

GO
